# 📊 Khám phá Dữ liệu Ride-Sharing (Kaggle Dataset)
**Mục tiêu EDA:**
1. Đọc và hiểu cấu trúc của 5 bảng dữ liệu (`users`, `rides`, `drivers`, `vehicles`, `ratings`).
2. Kiểm định từng thành phần: Có logic kinh doanh thực tế hay chỉ là dữ liệu ngẫu nhiên?
3. Rút ra bảng tham số (Parameters) để làm nền cho Mô phỏng Dữ liệu ở Tuần 3.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

## 1. Đọc dữ liệu & Tổng quan Schema

In [ ]:
DATA_DIR = '../../data/kaggle_ridesharing/'

df_users    = pd.read_csv(DATA_DIR + 'users.csv')
df_rides    = pd.read_csv(DATA_DIR + 'rides.csv')
df_drivers  = pd.read_csv(DATA_DIR + 'drivers.csv')
df_vehicles = pd.read_csv(DATA_DIR + 'vehicles.csv')
df_ratings  = pd.read_csv(DATA_DIR + 'ratings.csv')

print(f'Users   : {df_users.shape[0]:>7,} dòng | Cột: {list(df_users.columns)}')
print(f'Rides   : {df_rides.shape[0]:>7,} dòng | Cột: {list(df_rides.columns)}')
print(f'Drivers : {df_drivers.shape[0]:>7,} dòng | Cột: {list(df_drivers.columns)}')
print(f'Vehicles: {df_vehicles.shape[0]:>7,} dòng | Cột: {list(df_vehicles.columns)}')
print(f'Ratings : {df_ratings.shape[0]:>7,} dòng | Cột: {list(df_ratings.columns)}')

> **💡 Nhận xét:** Bộ dữ liệu có 10,000 users, 300 tài xế, 300 xe, 50,000 chuyến đi và 50,000 lượt đánh giá.
> Schema đạt chuẩn một nền tảng gọi xe thực thụ (tương tự Uber/Xanh SM):
> `Users ←→ Rides ←→ Drivers ←→ Vehicles` nối với nhau qua các khóa `user_id`, `driver_id`, `vehicle_id`.
> **✅ Kết luận:** Schema 5 bảng có thể tham khảo để thiết kế cấu trúc dữ liệu mô phỏng ở Tuần 3.

## 2. Hành vi người dùng: Số chuyến đi / User

In [ ]:
user_stats = df_rides.groupby('user_id').agg(
    total_rides  = ('ride_id', 'count'),
    total_spent  = ('fare_amount', 'sum'),
    avg_distance = ('distance_km', 'mean')
).reset_index()

plt.figure(figsize=(10, 5))
sns.histplot(user_stats['total_rides'], bins=30, kde=True, color='royalblue')
plt.title('Phân phối Số chuyến đi trên mỗi User', fontsize=14)
plt.xlabel('Số chuyến đi')
plt.ylabel('Số lượng User')
plt.axvline(user_stats['total_rides'].mean(), color='red', linestyle='--',
            label=f"Mean = {user_stats['total_rides'].mean():.1f}")
plt.legend()
plt.show()
print(user_stats['total_rides'].describe())

> **💡 Nhận xét:** Phân phối có dạng gần chuẩn (hình chuông), với đỉnh tập trung ở **4-6 chuyến/kỳ** và đuôi dài về phía phải.
> Điều này cho thấy phần lớn khách hàng là **Regular Users**, số lượng **Heavy Users (>10 chuyến)** khá ít.
> **✅ Tham khảo được:** Dùng `np.random.poisson(lam=5)` để mô phỏng số chuyến/user trong Tuần 3.

## 3. Chất lượng dịch vụ: Phân phối Điểm đánh giá

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df_ratings, x='rating_value', palette='viridis')
plt.title('Phân phối Điểm đánh giá (Rating Distribution)', fontsize=14)
plt.xlabel('Điểm số (1-5 sao)')
plt.ylabel('Số lượng đánh giá')
plt.show()

print(df_ratings['rating_value'].value_counts().sort_index())

> **⚠️ Nhận xét (Dấu hiệu bất thường):** Số lượng đánh giá ở mỗi mức điểm (1,2,3,4,5 sao) gần như **bằng nhau tăm tắp** (~10,000/mức).
> Trong thực tế (Uber, Grab, Xanh SM), phân phối Rating luôn có dạng **chữ J lệch phải**: Đa số là 5 sao, rất ít 1-2 sao.
> Điều này chứng tỏ cột `rating_value` được sinh ra bằng `random.randint(1, 5)` — hoàn toàn ngẫu nhiên.
> **❌ Không tham khảo:** Khi mô phỏng Tuần 3, tự thiết kế lại: `rating = f(received_voucher, vip_status) + noise`.

## 4. Tương quan: Lòng trung thành (Số chuyến) vs Sự hài lòng (Rating)

In [ ]:
user_ratings = df_ratings.groupby('user_id')['rating_value'].mean().reset_index()
user_ratings.rename(columns={'rating_value': 'avg_rating'}, inplace=True)
user_analysis = pd.merge(user_stats, user_ratings, on='user_id', how='left')
user_analysis['user_segment'] = pd.cut(
    user_analysis['total_rides'],
    bins=[0, 3, 8, 20],
    labels=['Occasional (1-3)', 'Regular (4-8)', 'Heavy (>8)']
)

plt.figure(figsize=(8, 5))
sns.boxplot(data=user_analysis, x='user_segment', y='avg_rating', palette='Set2')
plt.title('Điểm đánh giá Trung bình theo Tệp Khách hàng', fontsize=14)
plt.xlabel('Phân khúc Khách hàng (Dựa trên số chuyến)')
plt.ylabel('Điểm đánh giá TB (Sao)')
plt.show()

> **⚠️ Nhận xét (Xác nhận bất thường):** Boxplot của 3 phân khúc (Occasional, Regular, Heavy) gần như **giống hệt nhau** (Median ~3.0, IQR tương đương).
> Trong thực tế, Heavy Users thường cho rating cao hơn (vì họ đã quen và hài lòng với dịch vụ) hoặc thấp hơn (vì họ có kỳ vọng cao).
> Sự đồng nhất tuyệt đối này xác nhận: `rating` và `total_rides` hoàn toàn **không có quan hệ nhân quả** trong bộ data Kaggle.
> **❌ Không tham khảo:** Cần tự code logic Causal trong mô phỏng Tuần 3.

## 5. Kiểm chứng: Giá cước ~ Quãng đường (Có công thức hay ngẫu nhiên?)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_rides.sample(5000), x='distance_km', y='fare_amount',
                alpha=0.4, color='coral')
plt.title('Tương quan Quãng đường vs Giá cước (Mẫu 5,000 chuyến)', fontsize=14)
plt.xlabel('Quãng đường (km)')
plt.ylabel('Giá cước ($)')
plt.show()

corr = df_rides['distance_km'].corr(df_rides['fare_amount'])
print(f'Hệ số tương quan Pearson: {corr:.4f}')
print(f'Tỷ lệ ước tính: fare ≈ {df_rides["fare_amount"].mean() / df_rides["distance_km"].mean():.2f} $/km')

> **✅ Nhận xét (Tốt!):** Hệ số tương quan **r = 0.93** — cực kỳ cao. Biểu đồ hình nón (Heteroskedastic) cho thấy:
> - Giá cước **tăng tuyến tính** theo quãng đường — đúng với logic taxi thực tế.
> - Mức độ phân tán (nhiễu) tăng dần khi xa hơn — mô phỏng được ảnh hưởng của kẹt xe, thời gian chờ.
> **✅ Tham khảo được:** Dùng `fare = distance_km × 2.5 + noise` trong mô phỏng Tuần 3 (~$2.5/km).

## 6. Kiểm định Nhân khẩu học Users (Tuổi & Giới tính)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_users['age'], bins=20, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Phân phối Tuổi (Age Distribution)', fontsize=13)
axes[0].set_xlabel('Tuổi')
axes[0].axvline(df_users['age'].mean(), color='red', linestyle='--',
                label=f"Mean = {df_users['age'].mean():.1f}")
axes[0].legend()

gender_counts = df_users['gender'].value_counts()
axes[1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452', '#55A868'], startangle=90)
axes[1].set_title('Phân phối Giới tính', fontsize=13)

plt.tight_layout()
plt.show()
print(f'Tuổi TB: {df_users["age"].mean():.1f} | Min: {df_users["age"].min()} | Max: {df_users["age"].max()}')
print(f'Giới tính:\n{gender_counts}')

> **⚠️ Nhận xét:** Phân phối Tuổi gần như **phẳng (Uniform)** từ 18-65, không có đỉnh rõ ràng ở tuổi 25-35 như thực tế.
> Giới tính chia **xấp xỉ 33%/33%/33%** cho Male/Female/Other — đây là dấu hiệu rõ ràng của random hóa.
> Thực tế (Grab, Xanh SM): ~55% Nam, ~40% Nữ, ~5% Other; Tuổi đỉnh ở 22-35.
> **❌ Không tham khảo:** Tự thiết kế lại trong mô phỏng:
> `age ~ Normal(30, 8)` và `gender: [Male 55%, Female 40%, Other 5%]`

## 7. Kiểm định Phân phối Thời gian (Giờ cao điểm thực tế?)

In [ ]:
df_rides['ride_start_time'] = pd.to_datetime(df_rides['ride_start_time'])
df_rides['hour']        = df_rides['ride_start_time'].dt.hour
df_rides['day_of_week'] = df_rides['ride_start_time'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hourly = df_rides['hour'].value_counts().sort_index()
axes[0].bar(hourly.index, hourly.values, color='steelblue', alpha=0.8)
axes[0].set_title('Số chuyến đi theo Giờ trong ngày', fontsize=13)
axes[0].set_xlabel('Giờ (0-23)')
axes[0].set_ylabel('Số chuyến')
axes[0].set_xticks(range(0, 24))

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df_rides['day_of_week'].value_counts().reindex(day_order)
axes[1].bar(daily.index, daily.values, color='coral', alpha=0.8)
axes[1].set_title('Số chuyến đi theo Ngày trong tuần', fontsize=13)
axes[1].set_xlabel('Ngày')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

> **⚠️ Nhận xét:** Cả 2 biểu đồ đều **phẳng hoàn toàn** — không có giờ cao điểm (7-9h, 17-19h), không có ngày bận hơn (Thứ 6).
> Trong thực tế, nhu cầu đi xe có nhịp điệu rõ ràng: Sáng & chiều giờ cao điểm x3 lần so với giữa trưa.
> **❌ Không tham khảo:** Khi mô phỏng, tự code phân phối thời gian dựa trên Notebook 2 (Yellow Taxi thực tế):
> Lấy `hour` từ dữ liệu Taxi thật để làm Baseline Demand cho mô phỏng.

## 8. Kiểm định Rating Tài xế (Có Positivity Bias thực tế không?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_drivers['rating'], bins=20, kde=True, color='mediumpurple', ax=axes[0])
axes[0].set_title('Phân phối Rating Tài xế', fontsize=13)
axes[0].set_xlabel('Rating (1-5)')
axes[0].axvline(df_drivers['rating'].mean(), color='red', linestyle='--',
                label=f"Mean = {df_drivers['rating'].mean():.2f}")
axes[0].legend()

sns.histplot(df_drivers['total_rides'], bins=20, kde=True, color='seagreen', ax=axes[1])
axes[1].set_title('Phân phối Tổng số chuyến của Tài xế', fontsize=13)
axes[1].set_xlabel('Tổng số chuyến đã chạy')

plt.tight_layout()
plt.show()
print(f'Rating TB tài xế : {df_drivers["rating"].mean():.2f}')
print(f'Tỷ lệ available  : {df_drivers["available"].mean()*100:.1f}%')

> **⚠️ Nhận xét:** Rating TB tài xế = **2.98** — gần như đúng bằng 3.0 (Trung tâm của thang 1-5).
> Thực tế (Uber/Grab): Rating tài xế lệch mạnh về **4.5-4.8** (Positivity Bias — tài xế điểm thấp bị loại khỏi hệ thống).
> Tỷ lệ tài xế Available = 48.3% — gần 50/50, là dấu hiệu random hóa.
> **❌ Không tham khảo:** Tự thiết kế lại: `driver_rating ~ Normal(4.5, 0.3)` và `availability = f(hour_of_day)`

---
## 9. TỔNG KẾT: Bảng Kiểm định Tham số cho Mô phỏng Tuần 3

In [ ]:
summary_data = {
    'Tham số': [
        'Schema 5 bảng (ERD)',
        'Tương quan fare ~ distance',
        'Số chuyến/user (Poisson ~5)',
        'Phân phối Tuổi User',
        'Giới tính (33/33/33%)',
        'Giờ cao điểm',
        'Ngày trong tuần',
        'Rating khách (1-5 đều)',
        'Rating tài xế (mean~3)',
    ],
    'Kết quả Kiểm định': [
        '✅ CÓ LOGIC — Dùng được',
        '✅ CÓ LOGIC (r=0.93) — Dùng được',
        '✅ CÓ LOGIC (hình chuông) — Dùng được',
        '❌ RANDOM (Uniform 18-65)',
        '❌ RANDOM (33% đều nhau)',
        '❌ RANDOM (Phẳng 24/7)',
        '❌ RANDOM (Phẳng 7 ngày)',
        '❌ RANDOM (Phẳng 1-5 sao)',
        '❌ RANDOM (mean=2.98 ≈ 3.0)',
    ],
    'Hành động Tuần 3': [
        'Copy cấu trúc, thêm cột Treatment/Outcome',
        'fare = distance_km × 2.5 + noise',
        'np.random.poisson(lam=5)',
        'np.random.normal(loc=30, scale=8).clip(18,60)',
        'np.random.choice([M,F,O], p=[0.55,0.40,0.05])',
        'Dùng phân phối giờ từ Yellow Taxi (Notebook 2)',
        'Dùng phân phối ngày từ Yellow Taxi (Notebook 2)',
        'rating = 3.5 + 0.5×voucher - 0.3×is_vip + noise',
        'np.random.normal(loc=4.5, scale=0.3).clip(1,5)',
    ]
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)

> **📌 Kết luận cuối cùng:**
> Bộ dữ liệu Kaggle Ride-Sharing là dữ liệu **tổng hợp (Synthetic)** được tạo bằng `Faker`.
> Chỉ **3/9 tham số** có giá trị tham khảo trực tiếp (Schema, fare~distance, rides/user).
> **6/9 tham số còn lại** bị random hóa và phải tự thiết kế lại theo logic thực tế của ngành gọi xe.
> Việc phát hiện ra điều này là **kỹ năng cốt lõi** của Data Scientist: Không tin tưởng mù quáng vào bất kỳ nguồn dữ liệu nào mà không kiểm chứng.